# Export Ground Sensor Data

## About this Notebook

### Objective

Prepare a clean, structured ground sensor NO₂ dataset to serve as the modeling target for the satellite fusion pipeline.

This notebook standardizes raw sensor measurements, aligns them temporally, and exports an analysis-ready dataset for downstream modeling.

---

### Inputs

- Raw ground sensor NO₂ data (CSV)
- Sensor metadata (sensor ID, latitude, longitude)
- Configuration parameters (aggregation level, filtering rules)

Data location:
`gs://msads-mba-capstone-team-1/Data/`

---

### Outputs

- Cleaned and aggregated ground sensor NO₂ dataset
- One row per sensor per time period (e.g., quarter)
- Exported CSV saved to GCS for downstream notebooks

Example output path:
`gs://msads-mba-capstone-team-1/Data/TrainingData/ground_sensor_no2.csv`

This dataset is later merged with:
- Sentinel-2 image chips
- Sentinel-5P NO₂ data
- Facility-level and ADI data

---

### Workflow Overview

1. Load raw sensor data  
2. Clean and validate measurements  
3. Aggregate to the required temporal resolution  
4. Verify geospatial fields  
5. Export final dataset to GCS  

---

### Key Notes

- Ground sensor NO₂ values serve as the supervised learning target.
- Temporal aggregation must match the satellite data cadence.
- Schema consistency is critical for downstream fusion modeling.
- All exports should be reproducible and deterministic.

## Configuration

In [3]:
out_path = (
    "gs://msads-mba-capstone-team-1/"
    "Data/TrainingData/sensor_data_clean.csv"
)

## Setup

In [1]:
# 0. Imports
import pystac_client
import planetary_computer
import pandas as pd
import numpy as np

In [2]:
# 1. Connect to STAC
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [3]:
# 2. Get ALL Eclipse weekly items
items = list(
    catalog.search(collections=["eclipse"]).items()
)

## Get Data

In [4]:
# 3. Load and stack weekly sensor data

dfs = []
bad_items = []

for item in items:
    try:
        asset = next(
            a for a in item.assets.values()
            if a.media_type == "application/x-parquet"
        )

        df = pd.read_parquet(
            asset.href,
            engine="pyarrow",
            storage_options=asset.extra_fields["table:storage_options"]
        )

        dfs.append(df)

    except Exception as e:
        bad_items.append((item.id, str(e)))

data = pd.concat(dfs, ignore_index=True)

In [5]:
# 4. Parse timestamp
data["ReadingDateTimeUTC"] = pd.to_datetime(
    data["ReadingDateTimeUTC"], utc=True
)

In [6]:
data.head()

,City,DeviceId,LocationName,Latitude,Longitude,ReadingDateTimeUTC,PM25,CalibratedPM25,CalibratedO3,CalibratedNO2,CO,Temperature,Humidity,BatteryLevel,PercentBattery,CellSignal
0,Chicago,2004,California & 26th (SB),41.8451,-87.695204,2023-02-26 00:04:43+00:00,10.348934,12.12,15.78,19.06,0.206461,2.087860,63.171387,4.073906,89.732773,-78.0
1,Chicago,2004,California & 26th (SB),41.8451,-87.695204,2023-02-26 00:09:53+00:00,11.791849,14.26,13.17,22.53,0.280271,1.954346,63.429260,4.073594,89.590729,-79.0
2,Chicago,2004,California & 26th (SB),41.8451,-87.695204,2023-02-26 00:15:03+00:00,10.922678,12.42,15.13,25.34,0.227858,1.858215,63.310242,4.073594,89.590729,-78.0
3,Chicago,2004,California & 26th (SB),41.8451,-87.695204,2023-02-26 00:20:13+00:00,10.852129,11.84,18.88,18.28,0.179647,1.828842,64.112854,4.073437,89.590729,-78.0
4,Chicago,2004,California & 26th (SB),41.8451,-87.695204,2023-02-26 00:25:24+00:00,11.343349,13.36,14.30,21.61,0.253491,1.836853,64.024353,4.073125,89.448685,-76.0


## Data Manipulation and Cleaning

In [7]:
# 5. Add quarter (TRUE reading-time based)
data["quarter"] = data["ReadingDateTimeUTC"].dt.to_period("Q")
# Extract just the calendar date
data["date"] = data["ReadingDateTimeUTC"].dt.date

/var/tmp/ipykernel_819675/990307327.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["quarter"] = data["ReadingDateTimeUTC"].dt.to_period("Q")


In [8]:
# 6. Mean quarterly sensor reading (LABEL)
sensor_quarter = (
    data
    .groupby(["DeviceId", "quarter"], as_index=False)
    .agg(
        no2_mean=("CalibratedNO2", "mean"),
        pm25_mean=("CalibratedPM25", "mean"),
        o3_mean=("CalibratedO3", "mean"),
        num_days=("date", "nunique")   # ← counts unique days in that quarter
    )
)

In [9]:
# 7. Static sensor locations
sensors = (
    data[["DeviceId", "Latitude", "Longitude"]]
    .drop_duplicates()
)

In [10]:
sensors = sensors.drop_duplicates(subset=["DeviceId"])

In [11]:
sensor_quarter["num_days"].describe()

count    859.000000
mean      71.359721
std       29.816843
min        1.000000
25%       63.000000
50%       90.000000
75%       92.000000
max       92.000000
Name: num_days, dtype: float64

In [12]:
len(sensor_quarter[sensor_quarter["num_days"] >= 45])

700

In [13]:
# Keep sensor quarters with at least 45 days of readings
sensor_quarter = sensor_quarter[sensor_quarter["num_days"] >= 45]

In [14]:
sensor_counts = sensor_quarter.groupby("DeviceId").size()
sensor_counts.describe()

count    126.000000
mean       5.555556
std        1.920648
min        1.000000
25%        5.000000
50%        7.000000
75%        7.000000
max        7.000000
dtype: float64

In [15]:
# Keep sensors with at least 3 quarters
valid_sensors = sensor_counts[sensor_counts >= 3].index
len(valid_sensors)

111

In [16]:
sensor_quarter = sensor_quarter[sensor_quarter["DeviceId"].isin(valid_sensors)]

In [17]:
# 8. Final sensor × quarter table
sensors_final = sensors.merge(sensor_quarter, on="DeviceId")

In [18]:
# 9. Quarter boundaries (for Sentinel-2)
sensors_final["q_start"] = sensors_final["quarter"].dt.start_time.dt.date
sensors_final["q_end"]   = sensors_final["quarter"].dt.end_time.dt.date

## Sanity Checks

In [19]:
print(sensors_final.head(10))

   DeviceId  Latitude  Longitude quarter   no2_mean  pm25_mean    o3_mean  \
0      2004   41.8451 -87.695204  2021Q3   9.837126  13.072161  32.996590   
1      2004   41.8451 -87.695204  2021Q4  13.889216  10.791195  20.117447   
2      2004   41.8451 -87.695204  2022Q1  12.688283   9.802504  18.686535   
3      2004   41.8451 -87.695204  2022Q2  10.932224  10.035338  30.087279   
4      2004   41.8451 -87.695204  2022Q3   9.139607  10.753448  32.074869   
5      2004   41.8451 -87.695204  2022Q4  13.253729  10.956194  20.781666   
6      2004   41.8451 -87.695204  2023Q1  14.041194  11.549462  19.618346   
7      2005   41.8524 -87.695200  2021Q3   9.370114  13.699839  31.069497   
8      2005   41.8524 -87.695200  2022Q2  13.093310  10.685563  26.844144   
9      2005   41.8524 -87.695200  2022Q3  12.201211  11.217453  31.103987   

   num_days     q_start       q_end  
0        92  2021-07-01  2021-09-30  
1        92  2021-10-01  2021-12-31  
2        76  2022-01-01  2022-03-31  


In [20]:
#sanity checks
sensors_final.groupby("quarter").size()

quarter
2021Q3     94
2021Q4     97
2022Q1     97
2022Q2    105
2022Q3    108
2022Q4     93
2023Q1     85
Freq: Q-DEC, dtype: int64

In [21]:
#sanity checks
len(sensors_final.groupby("DeviceId"))

111

In [22]:
sensors_final.groupby(["DeviceId","quarter"]).size().sort_values(ascending=False).head()

DeviceId  quarter
2212      2023Q1     1
2002      2021Q3     1
          2021Q4     1
          2022Q1     1
          2022Q2     1
dtype: int64

In [23]:
sensors_final["no2_mean"].describe()

count    679.000000
mean      12.994251
std        4.657189
min        2.361252
25%        9.730874
50%       12.796327
75%       16.131519
max       26.486733
Name: no2_mean, dtype: float64

## Save Data

In [26]:
sensors_final["no2_log"] = np.log1p(sensors_final["no2_mean"])

In [4]:
# Output
sensors_final.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: gs://msads-mba-capstone-team-1/Data/TrainingData/sensor_data_clean.csv
